<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/MMLU_Human_Validation_Sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json
import random
import pandas as pd
from google.colab import files


# -------------------------
# Settings
# -------------------------

RANDOM_SEED = 42
SAMPLE_SIZE = 10
BENCHMARK_NAME = "MMLU"

FILE_PATHS = {
    "0.1": "/content/mmlu_rate_0.1.jsonl",
    "0.4": "/content/mmlu_rate_0.4.jsonl",
    "0.7": "/content/mmlu_rate_0.7.jsonl"
}


# -------------------------
# Read JSONL files
# -------------------------

def read_jsonl(file_path):
    records = []

    with open(file_path, "r", encoding="utf-8-sig") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
                record["_source_row"] = line_number
                records.append(record)

            except json.JSONDecodeError as error:
                print(
                    f"Invalid JSON on line {line_number} "
                    f"in {file_path}: {error}"
                )

    return records


# -------------------------
# Load all typo-rate datasets
# -------------------------

datasets = {}

for typo_rate, file_path in FILE_PATHS.items():
    datasets[typo_rate] = read_jsonl(file_path)

    print(
        f"Loaded {len(datasets[typo_rate])} records "
        f"for typo rate {typo_rate}"
    )


# -------------------------
# Create lookup tables
# -------------------------

record_maps = {}

for typo_rate, records in datasets.items():
    record_maps[typo_rate] = {
        record["original_text_backup"]: record
        for record in records
        if record.get("original_text_backup")
    }


# -------------------------
# Find prompts shared by all three rates
# -------------------------

common_original_prompts = set(record_maps["0.1"].keys())

for typo_rate in ["0.4", "0.7"]:
    common_original_prompts = common_original_prompts.intersection(
        record_maps[typo_rate].keys()
    )

common_original_prompts = sorted(common_original_prompts)

print(f"Common original prompts: {len(common_original_prompts)}")


if len(common_original_prompts) < SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(common_original_prompts)} matched prompts were found. "
        f"Cannot sample {SAMPLE_SIZE} prompts."
    )


# -------------------------
# Randomly sample original prompts
# -------------------------

random.seed(RANDOM_SEED)

selected_original_prompts = random.sample(
    common_original_prompts,
    SAMPLE_SIZE
)

print(f"Random seed: {RANDOM_SEED}")
print(f"Selected original prompts: {SAMPLE_SIZE}")


# -------------------------
# Build the annotation table
# -------------------------

output_rows = []

for prompt_number, original_prompt in enumerate(
    selected_original_prompts,
    start=1
):
    base_id = f"{BENCHMARK_NAME}_{prompt_number:03d}"

    for typo_rate in ["0.1", "0.4", "0.7"]:
        record = record_maps[typo_rate][original_prompt]

        output_rows.append({
            "Base_ID": base_id,
            "Sample_ID": f"{base_id}_rate_{typo_rate}",
            "Benchmark": BENCHMARK_NAME,
            "Typo_Rate": typo_rate,
            "Source_Row": record.get("_source_row", ""),
            "Original_Prompt": original_prompt,
            "Modified_Prompt": record.get("question", ""),
            "Gold_Answer": record.get("answer", ""),

            "R1_Meaning": "",
            "R2_Meaning": "",
            "Final_Meaning": "",

            "R1_Key_Info": "",
            "R2_Key_Info": "",
            "Final_Key_Info": "",

            "R1_Realism": "",
            "R2_Realism": "",
            "Final_Realism": "",

            "R1_Readability": "",
            "R2_Readability": "",
            "Final_Readability": "",

            "Comments": ""
        })


sample_df = pd.DataFrame(output_rows)

print(f"Total output rows: {len(sample_df)}")

display(sample_df)


# -------------------------
# Save the CSV file
# -------------------------

output_file = "/content/MMLU_matched_sample_n10_seed42.csv"

sample_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV saved successfully: {output_file}")


# -------------------------
# Download the CSV file
# -------------------------

files.download(output_file)

Loaded 14042 records for typo rate 0.1
Loaded 14042 records for typo rate 0.4
Loaded 14042 records for typo rate 0.7
Common original prompts: 13869
Random seed: 42
Selected original prompts: 10
Total output rows: 30


,Base_ID,Sample_ID,Benchmark,Typo_Rate,Source_Row,Original_Prompt,Modified_Prompt,Gold_Answer,R1_Meaning,R2_Meaning,...,R1_Key_Info,R2_Key_Info,Final_Key_Info,R1_Realism,R2_Realism,Final_Realism,R1_Readability,R2_Readability,Final_Readability,Comments
0,MMLU_001,MMLU_001_rate_0.1,MMLU,0.1,3756,"Unlike in a closed primary election, in an ope...","Unlike in a closed primary eleciton, in an ope...",2,,,...,,,,,,,,,,
1,MMLU_001,MMLU_001_rate_0.4,MMLU,0.4,3756,"Unlike in a closed primary election, in an ope...","Unlikje in a closed primary eleciton, in an op...",2,,,...,,,,,,,,,,
2,MMLU_001,MMLU_001_rate_0.7,MMLU,0.7,3756,"Unlike in a closed primary election, in an ope...","Unli in a close priamyr electiin, in an oen pr...",2,,,...,,,,,,,,,,
3,MMLU_002,MMLU_002_rate_0.1,MMLU,0.1,11515,A man was prosecuted for assault and battery a...,A man was prosecuted for assautl and battery a...,3,,,...,,,,,,,,,,
4,MMLU_002,MMLU_002_rate_0.4,MMLU,0.4,11515,A man was prosecuted for assault and battery a...,A man was prosecuted fort assasult and barrery...,3,,,...,,,,,,,,,,
5,MMLU_002,MMLU_002_rate_0.7,MMLU,0.7,11515,A man was prosecuted for assault and battery a...,A man was prosrcuted fir assault andf batteryt...,3,,,...,,,,,,,,,,
6,MMLU_003,MMLU_003_rate_0.1,MMLU,0.1,7169,These objectives are often the most suitable ...,These objectives are often tje most suitable w...,3,,,...,,,,,,,,,,
7,MMLU_003,MMLU_003_rate_0.4,MMLU,0.4,7169,These objectives are often the most suitable ...,Tnese objectives are often the mst suitable wh...,3,,,...,,,,,,,,,,
8,MMLU_003,MMLU_003_rate_0.7,MMLU,0.7,7169,These objectives are often the most suitable ...,These objecrive are often the most suitabwl wh...,3,,,...,,,,,,,,,,
9,MMLU_004,MMLU_004_rate_0.1,MMLU,0.1,7577,Which of the following has provided evidence t...,Which of the following had provided evidence t...,0,,,...,,,,,,,,,,


CSV saved successfully: /content/MMLU_matched_sample_n10_seed42.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>